## Assignment 3: $k$ Nearest Neighbor


**Q1.**
1. What is the difference between regression and classification?
2. What is a confusion table? What does it help us understand about a model's performance?
3. What does the SSE quantify about a particular model?
4. What are overfitting and underfitting? 
5. Why does splitting the data into training and testing sets, and choosing $k$ by evaluating accuracy or SSE on the test set, improve model performance?
6. With classification, we can report a class label as a prediction or a probability distribution over class labels. Please explain the strengths and weaknesses of each approach.

**Q2.** This is a case study on $k$ nearest neighbor classification, using the `land_mines.csv` data.

The data consists of a label, `mine_type`, taking integer values 1 to 5, and three properties of the mine, `voltage`, `height` and `soil`. We want to predict the kind of mine from data about it. Imagine working for the DOD or a humanitarian aid agency, trying to help people remove land mines more safely.

1. Load the data. Perform some EDA, summarizing the target label and the features.
2. Split the sample 50/50 into training and test/validation sets. (The smaller the data are, the more equal the split should be, in my experience: Otherwise, all of the members of one class end up in the training or test data, and the model falls apart.)
3. Build a $k$-NN classifier. Explain how you select $k$.
4. Print a confusion table for the optimal model, comparing predicted and actual class label on the test set. How accurate is it? Where is performance more or less accurate?
5. Notice that you can have a lot of accurate predictions for a given type of mine, but still make a lot of mistakes. Please explain how you'd advise someone to actually use this predictive model in practice, given the errors that it tends to make.

---

## *Phase 1: Solution without AI

Complete all questions in the following cells without generative AI. Then commit and push the
notebook before beginning Phase 2.

**Declaration:** I completed this version without generative AI.

**Student(s):** Aiden Zhu

**Date:** 9/30/2026

In [2]:
# Your Phase 1 solution to the problem goes here.
print("1) Regression is used to predict a numerical target while classification is used to predict a categorical or qualitative. In the example used in the lecture, regression could predict prices while classification could predict survival.")
print("2) A confusion table measures actual classes and predicted classes against each other. This allows us to understand the correctness of a model and determine the amount of correct predictions, or how well the model performs.")
print("3) SSE stands for Sum of Squared Error, and it calculates the prediction error of a validation set. A lower validation SSE means that the validation set has less errors.")
print("4) Overfitting and underfitting is determined by the value of k, which determines the number of close 'neighbors' that the model will look at to make a prediction. A high k value will create more overfit noise in the training data while a low k value could underfit noise and make predictions more similar.")
print("5) By splitting the data into training and testing sets, we can adjust models while comparing their predictions/outcomes on the testing set to evaluate model performance. We calculate the SSE or accuracy with different k values because it can show us the different training and validation errors for different k values and allows us to choose the lowest SSE value.")
print ("6) From the slides and vehicle example, I believe the advantage to using class labels is that we can see the actual vs predicted values from a clear numerical standpoint within each class, and it is easy to interpret individual predictions. The weakness is that it could be hard to interpret the overall model as the number of classes get larger. Also, it is a binary system that calculates only correct vs incorrect and doesn't include any confidence. The strength of a probability/prediction distribution is that it shows the confidence of the model rather than the binary right vs wrong. I think a weakness is that the distribution needs to be interpreted by people rather than straight up showing the right/wrong data, so conclusions can vary between people ")


1) Regression is used to predict a numerical target while classification is used to predict a categorical or qualitative. In the example used in the lecture, regression could predict prices while classification could predict survival.
2) A confusion table measures actual classes and predicted classes against each other. This allows us to understand the correctness of a model and determine the amount of correct predictions, or how well the model performs.
3) SSE stands for Sum of Squared Error, and it calculates the prediction error of a validation set. A lower validation SSE means that the validation set has less errors.
4) Overfitting and underfitting is determined by the value of k, which determines the number of close 'neighbors' that the model will look at to make a prediction. A high k value will create more overfit noise in the training data while a low k value could underfit noise and make predictions more similar.
5) By splitting the data into training and testing sets, we can 

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler       
from sklearn.neighbors import KNeighborsClassifier

df = pd.read_csv("data/land_mines.csv")
df.info()
print(df.isna().sum())
df.describe()
print(df["mine_type"].value_counts())
print("mine_type is a categorical variable with categories 1, 2, 3, 4, and 5. I was confused at first because the types are integers, but they are just labels for the mines. If they were called something else it would've been easier to differentiate. voltage, height, and soil are numbers between 0-1.")

eval_X_raw = df[["voltage", "height", "soil"]]
eval_y = df["mine_type"]
(
    eval_X_train_raw, 
    eval_X_valid_raw, 
    eval_y_train, 
    eval_y_valid 
) = train_test_split(
    eval_X_raw, eval_y, test_size=0.50, random_state=65
)

scaler = MinMaxScaler()
eval_X_train_scaled = scaler.fit_transform(eval_X_train_raw)
eval_X_valid_scaled = scaler.transform(eval_X_valid_raw)

eval_ks = np.arange(1, 51)
eval_valid_error = []
eval_train_error = []
for eval_k in eval_ks:
    eval_candidate = KNeighborsClassifier(n_neighbors=int(eval_k))
    eval_candidate.fit(eval_X_train_scaled, eval_y_train)
    eval_valid_hat = eval_candidate.predict(eval_X_valid_scaled)
    eval_train_hat = eval_candidate.predict(eval_X_train_scaled)
    eval_valid_error.append(np.mean(
        eval_y_valid.to_numpy() != eval_valid_hat
    ))
    eval_train_error.append(np.mean(
        eval_y_train.to_numpy() != eval_train_hat
    ))
eval_k_star = int(eval_ks[np.argmin(eval_valid_error)])
print(eval_k_star)
print()

print("To select k, I wanted to find the lowest error. I used error rate instead of SSE because mine_type is a categorical variable and not numerical. Mimicking the slides, I used k values from 1-50 on the training set then calculated the error rate on the validation set. To find the k with the lowest error, I used np.argmin like the slides did to find k=2 with a 45% accuracy rate. ")



model = KNeighborsClassifier(n_neighbors=eval_k_star).fit(eval_X_train_scaled, eval_y_train)
y_prediction = model.predict(eval_X_valid_scaled)

validation_set = eval_X_valid_raw.copy()
validation_set["actual"] = eval_y_valid
validation_set["predicted"] = y_prediction

for x in np.arange(1, 6):
    print(f"Actual type {x}:")
    print(validation_set.loc[validation_set["actual"] == x, "predicted"].value_counts())

print(1 - eval_valid_error[eval_k_star - 1])

print("The model has an accuracy of 45% and classifies 76 of 169 correctly. The performance is more accurate for mine_type 2 and less accurate for mine_type 4 and 5.")
print("The model is definitely better at predicted specific mine types than others, such as mine_type 2. Also, the model predicts mine_type 1 right many times, but then it mislabels as another mine_type so it gets it wrong. As a result, I wouldn't recommend using this model to make important final decisions because of its low accuracy on other types besides 2, but it performs pretty well for mine type 2. I would advise someone to not use the model until it is trained better, but it could be used to amke broad predictions because of its 45% accuracy in the meantime.")






<class 'pandas.DataFrame'>
RangeIndex: 338 entries, 0 to 337
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   voltage    338 non-null    float64
 1   height     338 non-null    float64
 2   soil       338 non-null    float64
 3   mine_type  338 non-null    int64  
dtypes: float64(3), int64(1)
memory usage: 10.7 KB
voltage      0
height       0
soil         0
mine_type    0
dtype: int64
mine_type
1    71
2    70
3    66
4    66
5    65
Name: count, dtype: int64
2

To select k, I wanted to find the lowest error. Mimicking the slides, I used k values from 1-50 on the training set then calculated the error rate on the validation set. To find the k with the lowest error, I used np.argmin like the slides did to find k=2 with a 45% accuracy rate. 
Actual type 1:
predicted
1    21
3     6
4     4
5     2
2     1
Name: count, dtype: int64
Actual type 2:
predicted
2    33
4     3
3     2
Name: count, dtype: int64
Actual type 3:


## *Phase 2: AI-assisted Revision
Keep Phase 1 frozen.

- **Phase 1 commit SHA ID:** 296aba0e074986bc4662c11adfb4c17972279e1d
- **AI tool and model used:** Claude Opus 5.5

### *Prompts used

1. For these first 6 questions, are my explanations right and are there any errors, if so list them out. respond in bullet points.
2. List how I can improve my code in bullet points and answer the questions better.

### *AI-proposed changes


[Paste the AI's concise numbered list verbatim.]

1. 1) Regression vs. classification: correct

Minor: "categorical or qualitative" is missing a noun. Write "categorical or qualitative target."
Optional addition: the type of target also changes how kNN predicts (regression averages the neighbors' outcomes, classification takes a majority vote) and how the model is evaluated (SSE vs. accuracy and the confusion matrix).

2) Confusion table: partly right, misses the key point

"Measures" should be "counts." Each cell counts observations by actual class (rows) and predicted class (columns).
Missing: diagonal cells are correct predictions, and off-diagonal cells are misclassifications.
Main gap: "the amount of correct predictions" is what accuracy already tells you. The confusion table's value is that it shows which classes the model confuses with which. In the vehicle example, SUVs and cars were often confused, and station wagons were never predicted. The Accuracy slide makes this point: accuracy doesn't show which classes are misclassified.

3) SSE: partly right, some imprecision

SSE isn't only for a validation set. It can be calculated on any data, including training data, as the slides do.
Say what it actually measures: the sum of the squared residuals (actual − predicted), the total size of the prediction errors. Squaring makes large misses count more.
"The validation set has less errors" should be "the model makes smaller errors on the validation set." SSE measures how big the errors are, not how many there are.
Add that SSE is for regression, a numeric target.

4) Overfitting and underfitting: has a major error

High and low k are reversed. A low k overfits (at k = 1 the model copies one neighbor's noise). A high k underfits (it averages so many neighbors that predictions become nearly the same for everyone).
The definitions are missing. Overfitting and underfitting aren't "determined by k." They're general ideas, and k is how they show up in kNN.
Overfitting: the model fits the noise in the training data, so it does well on training data but poorly on new data.
Underfitting: the model is too simple to capture the real pattern, so it does poorly on both.
"Underfit noise" is the wrong phrase. Underfitting means missing the signal, not fitting noise.

5) Train/test split: on track, missing the "why"

"Adjust models" is vague. Say you fit models on the training set and evaluate them on the held-out set.
Missing the core reason: training error is misleading. At k = 1, training error is zero because each point is its own neighbor, which rewards overfitting. Held-out data estimates how the model will do on new data.
"Choose the lowest SSE value" is right for SSE, but with accuracy you choose the highest. Say "lowest SSE or highest accuracy."
Optional: the slides call the set used to choose k the validation set and keep a separate test set for the final evaluation.
2. Q1: EDA

df.describe() doesn't display, because it's not the last line of the cell. Wrap it: print(df.describe()).
The summary is too informal and too thin. Cut "I was confused at first…" and the line about names. State facts instead:
338 mines, no missing values
mine_type: 5 categories, balanced (65–71 each, about 20% per type), so this is a classification problem
Features: all three already scaled 0–1, so no feature dominates the kNN distance. Voltage is skewed toward low values; height (12 levels) and soil (6 levels) are coded settings.
Mention that voltage is the most informative feature (type 2 is much higher), if you add code that shows it.

Q2: split

There's no written answer. Add one sentence: you split 50/50 with random_state=65 so the results are reproducible, and every mine type appears in both halves.
Show that it worked, which is your instructor's concern: print(eval_y_train.value_counts()) and print(eval_y_valid.value_counts()).

Q3: choosing k

Add why you used validation error instead of training error: at k = 1 the training error is 0% because each mine is its own neighbor, which rewards overfitting.
"Used k values from 1-50 on the training set" should be "fit a model for each k from 1–50 on the training set."
Optional: note that k = 11 performs almost as well (a one-mine difference), and that k = 2 can produce 1–1 tie votes.

Q4: confusion table

Include every type, not just 2, 4 and 5: type 2 at 87% correct, type 1 at 62%, type 3 at 33%, type 4 at 24%, type 5 at 10%.
Explain why type 2 does well: its voltage is much higher than the other types', and types 1, 3, 4 and 5 overlap.
Point out the over-prediction of type 1: the model predicts it 49 times but is right only 21 times.

Q5: advice

The type 1 sentence is backwards. The model doesn't predict type 1 right and then mislabel it. It labels many mines of other types as type 1: 28 of its 49 type 1 predictions are wrong.
It contradicts itself: "don't use the model until it's trained better" vs. "use it for broad predictions." Pick one clear recommendation.
45% accuracy isn't a reason to use the model. Remove that reasoning.
Give concrete advice: use it only for screening, never treat a prediction as proof a mine is safe, have a technician confirm every mine, trust type 2 predictions more than the others, and flag predictions where the neighbors' vote is split.
Typos: "predicted" should be "predicting," and "amke" should be "make."

### *Evaluation of AI suggestions


| Change | Decision | Reason and verification |
|---|---|---|
| 1 | modify| I took most of the reasons and added it onto my own, and I fixed one of my answers and some of my statements.|
| 2 |modify  | I took a lot of the AI's corrections and rewriting to my responses because it included a lot of information I previously missed. |

### *AI-assisted solution in full
After the evaluation of AI suggestions, incorporate the accepted revisions into your Phase 1 solution and provide the final Phase 2 solution in full in the following cells.

In [ ]:
# Your Phase 2 solution to the problem goes here.
print("1) Regression is used to predict a numerical target, while classification is used to predict a categorical or qualitative target. In the lecture's example, regression could predict prices while classification could predict survival. This also changes how kNN makes predictions: regression averages the outcomes of the k nearest neighbors, while classification takes a majority vote among them.")

print("2) A confusion table counts observations by their actual class (rows) and predicted class (columns). The diagonal cells are correct predictions, and the off-diagonal cells are misclassifications. Beyond overall accuracy, it shows which classes the model confuses with each other. For example, in the vehicle data SUVs and cars were often mixed up, and station wagons were never predicted at all, which accuracy alone would hide.")

print("3) SSE stands for Sum of Squared Error. It adds up the squared residuals (actual minus predicted) for every observation, so it measures the total size of a regression model's prediction errors. Squaring keeps positive and negative errors from canceling out and makes large misses count more. A lower validation SSE means the model makes smaller errors on data it wasn't fit on.")

print("4) Overfitting is when a model fits the noise in the training data, so it performs well on the training data but poorly on new data. Underfitting is when a model is too simple to capture the real pattern, so it performs poorly on both. In kNN, a low k overfits: with k = 1, each prediction copies a single neighbor, noise included. A high k underfits: averaging many neighbors makes predictions nearly the same for every observation.")

print("5) Training error alone is misleading: with k = 1, each training point is its own nearest neighbor, so training error is zero even though the model is just memorizing noise. By splitting the data, we fit models on the training set and evaluate them on the held-out set, which the models never saw. That error estimates how well each model will predict new data. We calculate SSE (regression) or accuracy (classification) on the held-out set for each value of k and choose the k with the lowest SSE or highest accuracy, which balances overfitting and underfitting. The slides call the set used to choose k a validation set and keep a separate test set for the final evaluation.")

print("6) A class label gives one clear prediction, which makes it simple to act on and easy to evaluate with accuracy and a confusion table. Its weakness is that it hides the model's confidence: a close 5-4 vote looks the same as a unanimous 9-0 vote, and rare classes can disappear, like station wagons, which were never predicted. A probability distribution (the share of the k neighbors in each class) shows how confident the model is, keeps rare classes visible, and lets us set our own decision threshold when some mistakes are more costly than others. Its weaknesses are that it's harder to interpret, it doesn't give a decision on its own (someone still has to turn it into one), and with kNN the probabilities are coarse: they can only be multiples of 1/k.")

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier

# ---------- Q1: Load the data and EDA ----------
df = pd.read_csv("data/land_mines.csv")
df.info()
print(df.isna().sum())
print(df.describe())
print(df["mine_type"].value_counts())

for x in np.arange(1, 6):
    print(f"Mean voltage for type {x}:", np.mean(df.loc[df["mine_type"] == x, "voltage"]))

print("1) The data have 338 mines and 4 columns, with no missing values. The target, mine_type, is a categorical variable with 5 classes (1-5). It is stored as integers, but the numbers are only labels for the kinds of mine, so this is a classification problem. The classes are balanced, with 65-71 mines each (about 20% per type). The features voltage, height, and soil are all numeric and already range from 0 to 1, so no single feature dominates the distance calculation in kNN. Voltage is skewed toward low values, while height (12 levels) and soil (6 levels) take only a few evenly spaced values. Voltage is the most informative feature: type 2 mines average about 0.72, while the other types average 0.30-0.40.")

# ---------- Q2: 50/50 split ----------
eval_X_raw = df[["voltage", "height", "soil"]]
eval_y = df["mine_type"]
(
    eval_X_train_raw,
    eval_X_valid_raw,
    eval_y_train,
    eval_y_valid
) = train_test_split(
    eval_X_raw, eval_y, test_size=0.50, random_state=65
)

print(eval_y_train.value_counts())
print(eval_y_valid.value_counts())

print("2) I split the data 50/50 into training and validation sets, using random_state=65 so the split is reproducible. Every mine type appears in both halves (28-37 per type in training and 29-38 in validation), so no class ends up entirely in one set.")

# ---------- Q3: kNN classifier and choosing k ----------
scaler = MinMaxScaler()
eval_X_train_scaled = scaler.fit_transform(eval_X_train_raw)
eval_X_valid_scaled = scaler.transform(eval_X_valid_raw)

eval_ks = np.arange(1, 51)
eval_valid_error = []
eval_train_error = []
for eval_k in eval_ks:
    eval_candidate = KNeighborsClassifier(n_neighbors=int(eval_k))
    eval_candidate.fit(eval_X_train_scaled, eval_y_train)
    eval_valid_hat = eval_candidate.predict(eval_X_valid_scaled)
    eval_train_hat = eval_candidate.predict(eval_X_train_scaled)
    eval_valid_error.append(np.mean(
        eval_y_valid.to_numpy() != eval_valid_hat
    ))
    eval_train_error.append(np.mean(
        eval_y_train.to_numpy() != eval_train_hat
    ))
eval_k_star = int(eval_ks[np.argmin(eval_valid_error)])
print(eval_k_star)
print(eval_train_error[0], eval_valid_error[0])

print("3) To select k, I wanted the k with the lowest error on data the model wasn't fit on. Following the slides, I split, then scaled the features using only the training data. Then I fit a kNN classifier for each k from 1 to 50 on the training set and calculated its error rate (the share of mines predicted wrong) on the validation set. I used error rate instead of SSE because mine_type is categorical, so each prediction is simply right or wrong. I didn't use training error because it rewards overfitting: at k = 1 each mine is its own nearest neighbor, so training error is 0% while validation error is about 59%. Using np.argmin like the slides, the lowest validation error was at k = 2, with an error rate of 55% (45% accuracy). k = 11 performed almost as well (a difference of one mine), so a different split could pick a different k.")

# ---------- Q4: Confusion table ----------
model = KNeighborsClassifier(n_neighbors=eval_k_star).fit(eval_X_train_scaled, eval_y_train)
y_prediction = model.predict(eval_X_valid_scaled)

validation_set = eval_X_valid_raw.copy()
validation_set["actual"] = eval_y_valid
validation_set["predicted"] = y_prediction

for x in np.arange(1, 6):
    print(f"Actual type {x}:")
    print(validation_set.loc[validation_set["actual"] == x, "predicted"].value_counts())

print(1 - eval_valid_error[eval_k_star - 1])

print("4) The model classifies 76 of 169 validation mines correctly, an accuracy of 45%. That is better than the 20% expected from guessing among 5 types, but more than half the mines are misclassified. Performance is best for type 2 (33 of 38 correct, 87%), because type 2 mines have much higher voltage than the other types. It is moderate for type 1 (21 of 34, 62%) and poor for types 3 (10 of 30, 33%), 4 (9 of 38, 24%), and 5 (3 of 29, 10%), which overlap heavily in voltage and height. Many type 3, 4, and 5 mines are mislabeled as type 1.")

# ---------- Q5: Advice ----------
print("5) The model gets many type 1 mines right, but it also labels many mines of other types as type 1: it predicts type 1 49 times, and only 21 of those are correct, so a type 1 prediction is wrong 57% of the time. Only its type 2 predictions are fairly reliable (33 of 45 correct). I would advise using it only as a screening tool, not to make final decisions. A prediction should never be treated as proof that a mine is safe, and a trained technician should confirm every mine. Type 2 predictions could help with planning, but predictions of types 1, 3, 4, or 5 should be treated as uncertain, and mines where the neighbors' vote is split should get extra caution. Before relying on it more, the model would need more data or better features, because voltage, height, and soil can't separate types 1, 3, 4, and 5.")

<class 'pandas.DataFrame'>
RangeIndex: 338 entries, 0 to 337
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   voltage    338 non-null    float64
 1   height     338 non-null    float64
 2   soil       338 non-null    float64
 3   mine_type  338 non-null    int64  
dtypes: float64(3), int64(1)
memory usage: 10.7 KB
voltage      0
height       0
soil         0
mine_type    0
dtype: int64
          voltage      height        soil   mine_type
count  338.000000  338.000000  338.000000  338.000000
mean     0.430634    0.508876    0.503550    2.952663
std      0.195819    0.306043    0.344244    1.419703
min      0.197734    0.000000    0.000000    1.000000
25%      0.309737    0.272727    0.200000    2.000000
50%      0.359516    0.545455    0.600000    3.000000
75%      0.482628    0.727273    0.800000    4.000000
max      0.999999    1.000000    1.000000    5.000000
mine_type
1    71
2    70
3    66
4    66
5    65
Name: c

### *Reflection on AI assistance
In your own words without generative AI.

[In 150–300 words, describe the main improvements, AI mistakes or limitations, how the final solution was verified, and any remaining concerns.]

Honestly, the AI was able to capture a lot of information I missed from the kNN slideshow because there was so much content within it, so the AI was able to recognize what I missed and include it, helping me save some potential points that I missed. It was able to fix a lot of my open responses where I missed what the data was showing or other information I should have included. This assignment took a lot of piecing together parts of a giant idea which was really hard to approach. I think I'm definitely getting better at prompting the AI and getting exactly what I want from it. If I ask it something broad, it will not get it right, so I would rather take the time to write out exactly what I want than wait 30 seconds for it to respond something that isn't useful at all. 